
# Fig. 5b — Annotation Decoder cell-type-span cross-attention
## Standard Wilcoxon marker reference | new TRAIN / VAL split

本 notebook 保持 Stage2 Annotation Decoder 的原始自然语言任务不变，并将 marker reference 升级为标准单细胞差异表达定义。

```text
new Stage2 TRAIN
    ↓
one-vs-rest Wilcoxon rank-sum
    ↓
BH-FDR + logFC + pct_in/pct_out
    ↓
Top 30 train-derived cell-type markers

new Stage2 VAL
    ↓
Frozen scKITE Encoder + Annotation Decoder
    ↓
teacher-force 原始 natural_language_annotation
    ↓
精确定位真实 cell_type span
    ↓
最后一层 decoder→encoder cross-attention
    ↓
marker genes vs expression/detection-matched non-marker genes
    ↓
cell median → cell-type median → paired Wilcoxon
```

### 与旧版相比

- 使用新的 **TRAIN / VAL** 划分；不再使用旧 test。
- 不再以 `VAL >= 20 cells` 预筛 cell type。
- Marker 完全由 TRAIN 定义：Wilcoxon + BH-FDR + logFC + detection fraction。
- 每个 VAL cell 至少要有 5 个可匹配 marker 才进入 cell-level 分析。
- 最终统计阶段才要求每个 cell type 至少有 5 个 evaluable VAL cells。
- Statistical unit = **cell type**。


In [ ]:





from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'sckite').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'sckite').is_dir():
    raise FileNotFoundError('Run this notebook from inside the scKITE repository.')
STAGE2_YAML = REPO_ROOT / 'sckite' / 'stage2' / 'config.yaml'
STAGE2_CKPT = REPO_ROOT / 'checkpoints' / 'stage2' / 'best.pt'

PRETRAIN_TRAIN = None
PRETRAIN_VAL = None

OUT_DIR = REPO_ROOT / 'outputs' / 'biological_interpretation' / 'cell_type_span_attention'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CELL_ID_FIELD = 'cell_id'
CELL_TYPE_FIELD = 'cell_type'
ANNOTATION_FIELD = 'natural_language_annotation'
GENES_FIELD = 'genes'
EXPRESSIONS_FIELD = 'expressions'

SEED = 2026



ATTENTION_BATCH_SIZE = 110
MAX_VAL_CELLS = None
MIN_VAL_EVALUABLE_CELLS_PER_TYPE = 5


MIN_TRAIN_CELLS_PER_TYPE = 50
MARKER_TOP_N = 30
MARKER_FDR_MAX = 0.05
MARKER_LOGFC_MIN = 0.25
MARKER_PCT_IN_MIN = 0.10
REQUIRE_PCT_IN_GT_OUT = True



DE_TARGET_MAX_CELLS_PER_TYPE = 100



DE_REST_MAX_CELLS = 500



BACKGROUND_STATS_MAX_CELLS = 10000


DE_GENE_BATCH_SIZE = 1024


MATCH_N_BINS = 5
MATCH_REPEATS = 20
MIN_MATCHABLE_MARKERS_PER_CELL = 5


ANALYSIS_MLM_PROBABILITY = 0.0
USE_ORIGINAL_FIXED_EVAL_MASKING = False
EXCLUDE_EXPR_MASKED_GENES_FROM_ATTENTION = False



RUN_GENERATION_SANITY = False
N_GENERATION_EXAMPLES = 12
GEN_MAX_NEW_TOKENS = 64



SAVE_GENE_LEVEL_ATTENTION = False

FORCE_RESCAN = False
FORCE_RERUN_MARKERS = False
FORCE_RERUN_ATTENTION = False

WILCOXON_ALTERNATIVE = 'two-sided'

print('OUT_DIR:', OUT_DIR)


In [ ]:





import os, sys, re, json, math, random, zlib, warnings, heapq
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from scipy import sparse
from scipy.stats import rankdata, wilcoxon, mannwhitneyu
from tqdm.auto import tqdm

try:
    from streaming import StreamingDataset
except Exception as exc:
    raise ImportError('无法导入 streaming.StreamingDataset。') from exc

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sckite.stage2.tokenizer import GlobalGeneTextTokenizer
from sckite.stage2.data import Stage2Collator
from sckite.stage2.model import ScKITEStage2Model

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:





with open(STAGE2_YAML, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

if PRETRAIN_TRAIN is None:
    PRETRAIN_TRAIN = Path(cfg['paths']['train_local'])
else:
    PRETRAIN_TRAIN = Path(PRETRAIN_TRAIN)

if PRETRAIN_VAL is None:
    PRETRAIN_VAL = Path(cfg['paths']['val_local'])
else:
    PRETRAIN_VAL = Path(PRETRAIN_VAL)

gv_cfg = cfg['global_vocab']
data_cfg = cfg['data']
decoder_cfg = cfg['decoder']

tokenizer = GlobalGeneTextTokenizer.from_files(
    text_tokenizer_path=gv_cfg['text_tokenizer_path'],
    global_vocab_path=gv_cfg['global_vocab_path'],
    global_vocab_meta_path=gv_cfg['global_vocab_meta_path'],
    gene_table_path=gv_cfg['gene_table_path'],
    max_length=int(decoder_cfg.get('max_decoder_length', 512)),
    use_fast=bool(gv_cfg.get('use_fast', True)),
    do_lower_case=bool(gv_cfg.get('do_lower_case', False)),
)

model_cfg = dict(cfg['model'])
for k in ['stage1_ckpt_path','freeze_encoder','freeze_embeddings','freeze_value_head','gene_vocab_size','text_vocab_size','vocab_size']:
    model_cfg.pop(k, None)
model_cfg['global_vocab_size'] = int(tokenizer.vocab_size)
model_cfg['pad_token_id'] = int(tokenizer.pad_token_id)
model_cfg['mask_value'] = float(data_cfg.get('mask_value', -3.0))
model_cfg['max_decoder_length'] = int(decoder_cfg.get('max_decoder_length', 512))

model = ScKITEStage2Model(**model_cfg)
ckpt = torch.load(STAGE2_CKPT, map_location='cpu')
state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
if state and all(str(k).startswith('module.') for k in state):
    state = {str(k)[7:]: v for k,v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
critical_missing = [x for x in missing if x.startswith(('encoder','shared_token_embedding','value_encoder','annotation_decoder','annotation_lm_head'))]
if critical_missing:
    raise RuntimeError('Checkpoint missing critical parameters:\n' + '\n'.join(critical_missing[:100]))

model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)

print('TRAIN MDS:', PRETRAIN_TRAIN)
print('VAL MDS  :', PRETRAIN_VAL)
print('all frozen:', all(not p.requires_grad for p in model.parameters()))


In [ ]:











annotation_decoder_tasks = [
    {
        "name": "annotation",
        "field": ANNOTATION_FIELD,
        "target_type": "plain_text",
        "enabled": True,
    }
]

analysis_mlm_probability = (
    float(data_cfg.get("mlm_probability", 0.10))
    if USE_ORIGINAL_FIXED_EVAL_MASKING
    else 0.0
)

collator = Stage2Collator(
    tokenizer=tokenizer,
    max_encoder_length=int(data_cfg.get("max_encoder_length", 2049)),
    max_decoder_length=int(
        decoder_cfg.get(
            "max_decoder_length",
            data_cfg.get("max_decoder_length", 512),
        )
    ),
    mlm_probability=analysis_mlm_probability,
    num_bins=int(data_cfg.get("num_bins", 51)),
    sampling=bool(data_cfg.get("sampling", False)),
    keep_first_n_tokens=int(data_cfg.get("keep_first_n_tokens", 1)),
    pad_value=float(data_cfg.get("pad_value", -2.0)),
    cls_value=float(data_cfg.get("cls_value", -1.0)),
    mask_value=float(data_cfg.get("mask_value", -3.0)),
    mask_gene_input=bool(data_cfg.get("mask_gene_input", False)),
    mask_expr_input=bool(data_cfg.get("mask_expr_input", True)),
    decoder_tasks=annotation_decoder_tasks,
    genes_field=str(data_cfg.get("genes_field", GENES_FIELD)),
    expressions_field=str(data_cfg.get("expressions_field", EXPRESSIONS_FIELD)),
    loss_weight_field=data_cfg.get("loss_weight_field", "loss_weight"),
    use_loss_weight=False,
    task_sampling_mode="all",
    regulon_target_path=cfg["paths"]["regulon_target_path"],
    active_regulon_field=str(
        data_cfg.get("active_regulon_field", "active_regulon_ids")
    ),
    cell_id_field=str(data_cfg.get("cell_id_field", CELL_ID_FIELD)),
    n_regulons=int(data_cfg.get("n_regulons", 530)),
    regulon_num_queries=int(data_cfg.get("regulon_num_queries", 3)),
    regulon_sampling_mode=str(
        data_cfg.get("regulon_sampling_mode", "cyclic_without_replacement")
    ),
    regulon_seed=int(data_cfg.get("regulon_seed", 42)),
    validation_regulon_seed=int(
        data_cfg.get("validation_regulon_seed", 2026)
    ),
    cell_specific_targets=bool(
        data_cfg.get("cell_specific_targets", True)
    ),
    expressed_gene_threshold=float(
        data_cfg.get("expressed_gene_threshold", 0.0)
    ),
    dynamic_target_budget=bool(
        data_cfg.get("dynamic_target_budget", True)
    ),
    is_training=False,
    fixed_eval_encoder_mask=bool(
        data_cfg.get("fixed_eval_encoder_mask", True)
    ),
)

print("analysis mlm_probability:", collator.mlm_probability)
print("sampling:", collator.sampling)
print("fixed_eval_encoder_mask:", collator.fixed_eval_encoder_mask)
print("mask_gene_input:", collator.mask_gene_input)
print("mask_expr_input:", collator.mask_expr_input)


In [ ]:





train_ds = StreamingDataset(local=str(PRETRAIN_TRAIN), shuffle=False, batch_size=1, allow_unsafe_types=True)
val_ds = StreamingDataset(local=str(PRETRAIN_VAL), shuffle=False, batch_size=1, allow_unsafe_types=True)

print('TRAIN cells:', len(train_ds))
print('VAL cells  :', len(val_ds))

x0 = val_ds[0]
print('VAL keys:', sorted(x0.keys()))
required = [CELL_ID_FIELD, CELL_TYPE_FIELD, ANNOTATION_FIELD, GENES_FIELD, EXPRESSIONS_FIELD]
missing_fields = [x for x in required if x not in x0]
if missing_fields:
    raise KeyError(f'MDS missing fields: {missing_fields}')



## 5. 精确定位 annotation 中的 cell-type span

这里只允许：

> **case-insensitive exact substring match**

例如：

```text
cell_type:
memory B cell

annotation:
Activated memory B cell from a 26-year-old male human, isolated from blood.
          └───────────┘
```

会定位 `memory B cell`。

如果 cell type 在 annotation 中不存在精确文本对应：

- 不做 synonym mapping
- 不做 fuzzy matching
- 该 cell 从主机制分析中排除
- 单独报告 exact-span coverage

这样不会人为解释 annotation。


In [ ]:





def normalize_spaces(text):
    text = "" if text is None else str(text)
    text = text.replace("\u00A0", " ")
    return " ".join(text.split())

def find_subsequence(sequence, subsequence):
    sequence = list(sequence)
    subsequence = list(subsequence)

    if not subsequence or len(subsequence) > len(sequence):
        return []

    hits = []
    n = len(subsequence)

    for i in range(len(sequence) - n + 1):
        if sequence[i:i+n] == subsequence:
            hits.append(i)

    return hits

def locate_celltype_span(cell_type, annotation):
    """
    返回：
      success
      matched_surface_text
      char_start / char_end
      body_token_start / body_token_end
      decoder_query_positions

    decoder query position 与 target label position相同：
      decoder_input = [BOS] + target[:-1]
      logits[position] predicts target[position]
    """
    cell_type = normalize_spaces(cell_type)
    annotation = normalize_spaces(annotation)

    if not cell_type or not annotation:
        return {
            "success": False,
            "reason": "empty_celltype_or_annotation",
        }

    m = re.search(
        re.escape(cell_type),
        annotation,
        flags=re.IGNORECASE,
    )

    if m is None:
        return {
            "success": False,
            "reason": "no_exact_substring",
        }

    matched_surface = annotation[m.start():m.end()]

    body_ids = tokenizer.encode_plain_text_to_global_ids(annotation)
    span_ids = tokenizer.encode_plain_text_to_global_ids(matched_surface)

    hits = find_subsequence(body_ids, span_ids)

    if len(hits) == 0:
        return {
            "success": False,
            "reason": "text_match_but_token_match_failed",
        }



    body_start = int(hits[0])
    body_end = body_start + len(span_ids)

    task_name_ids = tokenizer.encode_task_name_to_global_ids("annotation")



    body_offset = 1 + len(task_name_ids) + 1

    query_positions = list(
        range(
            body_offset + body_start,
            body_offset + body_end,
        )
    )

    io = tokenizer.build_generic_task_decoder_io(
        task_name="annotation",
        content=annotation,
        target_type="plain_text",
        max_length=int(model.max_decoder_length),
    )

    if not query_positions:
        return {
            "success": False,
            "reason": "empty_query_positions",
        }

    if max(query_positions) >= len(io["decoder_input_ids"]):
        return {
            "success": False,
            "reason": "celltype_span_truncated_by_decoder_max_length",
        }

    return {
        "success": True,
        "reason": "ok",
        "matched_surface_text": matched_surface,
        "char_start": int(m.start()),
        "char_end": int(m.end()),
        "body_token_start": int(body_start),
        "body_token_end": int(body_end),
        "n_celltype_tokens": int(len(span_ids)),
        "n_token_occurrences": int(len(hits)),
        "decoder_query_positions": query_positions,
    }


for i in range(min(10, len(val_ds))):
    r = val_ds[i]
    info = locate_celltype_span(
        r[CELL_TYPE_FIELD],
        r[ANNOTATION_FIELD],
    )

    print("=" * 100)
    print("cell_type :", r[CELL_TYPE_FIELD])
    print("annotation:", r[ANNOTATION_FIELD])
    print("span info :", info)


In [ ]:





span_index_path = OUT_DIR / 'val_celltype_span_index.csv'
span_summary_path = OUT_DIR / 'val_celltype_span_summary.csv'

def scan_val_spans():
    n_total = len(val_ds) if MAX_VAL_CELLS is None else min(len(val_ds), int(MAX_VAL_CELLS))
    rows=[]
    for i in tqdm(range(n_total), desc='scan VAL cell-type spans'):
        rec=val_ds[i]
        ct=normalize_spaces(rec.get(CELL_TYPE_FIELD,''))
        ann=normalize_spaces(rec.get(ANNOTATION_FIELD,''))
        info=locate_celltype_span(ct,ann)
        rows.append({
            'dataset_index':i,
            'cell_id':str(rec.get(CELL_ID_FIELD,i)),
            'cell_type':ct,
            'natural_language_annotation':ann,
            'span_success':bool(info.get('success',False)),
            'span_reason':info.get('reason',''),
            'matched_surface_text':info.get('matched_surface_text',''),
            'n_celltype_tokens':info.get('n_celltype_tokens',0),
            'n_token_occurrences':info.get('n_token_occurrences',0),
        })
    return pd.DataFrame(rows)

if span_index_path.exists() and not FORCE_RESCAN:
    span_df=pd.read_csv(span_index_path)
else:
    span_df=scan_val_spans()
    span_df.to_csv(span_index_path,index=False)

val_exact_type_counts=(span_df[span_df['span_success']].groupby('cell_type').size().sort_values(ascending=False))
marker_candidate_cell_types=sorted(val_exact_type_counts.index.tolist())

span_summary=pd.DataFrame({
    'metric':['n_val_cells_scanned','n_exact_span_success','exact_span_coverage','n_cell_types_total','n_cell_types_with_exact_span'],
    'value':[len(span_df),int(span_df['span_success'].sum()),float(span_df['span_success'].mean()),int(span_df['cell_type'].nunique()),int(len(marker_candidate_cell_types))]
})
span_summary.to_csv(span_summary_path,index=False)
display(span_summary)
print('VAL exact-span cell types:',len(marker_candidate_cell_types))



## 7. Fast Wilcoxon marker reference

全量 `358,134 cells × 340 cell types × 全基因` 的 Scanpy Wilcoxon 非常慢，而且在几十万细胞下 P 值会被极大样本量主导。

这一版固定使用 **deterministic subsampled one-vs-rest Wilcoxon**：

```text
TRAIN 全量 metadata scan
        ↓
每个 cell type 最多保留 200 cells
        +
固定 1,000-cell global rest reservoir
        +
20,000-cell global background-stat reservoir
        ↓
raw counts → normalize_total(1e4) → log1p
        ↓
对于每个 cell type：
  target ≤200 cells
  rest ≤1000 cells（排除该类型）
        ↓
只对 pct_in ≥10% 的 genes 运行
Mann–Whitney U / Wilcoxon rank-sum
        ↓
BH-FDR 按完整 gene universe 校正
        ↓
FDR <0.05
log2FC >0.25
pct_in ≥0.10
pct_in > pct_out
        ↓
按 U/(n_in×n_out) effect size + logFC 排序
        ↓
Top30
```

### 为什么这样更适合这里

- `Mann–Whitney U` 与两独立样本的 Wilcoxon rank-sum 是等价检验。
- target 和 rest 的 cell 数在分析前固定，不根据最终 attention 结果调整。
- FDR 对完整 global gene universe 进行校正：未检验/未达到最低 detection 的 genes 视为 `P=1`，因此不会因为 detection prefilter 人为放宽 FDR。
- 20,000-cell reservoir 只用于估计 expression/detection matching bins；5×5 粗分箱不需要遍历全部 35.8 万 cells。
- 这版不再构建巨大的全量 `AnnData`，也不调用 `scanpy.tl.rank_genes_groups`。


In [ ]:





all_gene_ids = sorted(
    int(gid)
    for gid in tokenizer.global_id_to_token.keys()
    if tokenizer.global_id_is_gene(int(gid))
)

gene_id_to_col = {
    gid: i for i, gid in enumerate(all_gene_ids)
}
col_to_gene_id = np.asarray(all_gene_ids, dtype=np.int64)

def gene_name(gid):
    gid = int(gid)
    x = tokenizer.global_id_to_gene_symbol.get(gid)
    if x is not None and str(x).strip():
        return str(x)

    x = tokenizer.global_id_to_ensembl.get(gid)
    if x is not None and str(x).strip():
        return str(x)

    return tokenizer.global_id_to_token.get(gid, str(gid))

gene_names = np.asarray(
    [gene_name(gid) for gid in all_gene_ids],
    dtype=object,
)

print("number of gene tokens:", len(all_gene_ids))


In [ ]:





marker_all_path = OUT_DIR / 'train_fast_wilcoxon_markers_all.csv.gz'
marker_path = OUT_DIR / 'train_fast_wilcoxon_reference_markers_topN.csv'
train_gene_stats_path = OUT_DIR / 'train_sampled_global_gene_stats.csv'
selection_path = OUT_DIR / 'train_fast_wilcoxon_sampling_summary.csv'


def _hash32(text):
    return int(zlib.crc32(str(text).encode('utf-8')) & 0xffffffff)


def bh_adjust_full_universe(p_tested, tested_cols, n_total_genes):
    """
    BH-FDR across the complete gene universe.
    Genes not tested because pct_in < threshold are assigned p=1.
    """
    p_all = np.ones(int(n_total_genes), dtype=np.float64)
    p_all[np.asarray(tested_cols, dtype=np.int64)] = np.asarray(p_tested, dtype=np.float64)

    order = np.argsort(p_all, kind='mergesort')
    ranked = p_all[order]
    m = float(len(ranked))
    adj = ranked * m / np.arange(1, len(ranked) + 1, dtype=np.float64)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0.0, 1.0)

    out = np.empty_like(adj)
    out[order] = adj
    return out[np.asarray(tested_cols, dtype=np.int64)]


def aggregate_duplicate_gene_counts(cols, vals):
    cols = np.asarray(cols, dtype=np.int64)
    vals = np.asarray(vals, dtype=np.float64)
    if len(cols) == 0:
        return cols, vals
    u, inv = np.unique(cols, return_inverse=True)
    if len(u) == len(cols):
        return cols, vals
    out = np.zeros(len(u), dtype=np.float64)
    np.add.at(out, inv, vals)
    return u, out


def push_smallest_hash(heap, max_n, h, idx):
    """Keep deterministic smallest hashes using a max-heap implemented with -hash."""
    import heapq
    item = (-int(h), int(idx))
    if len(heap) < int(max_n):
        heapq.heappush(heap, item)
    elif item > heap[0]:
        heapq.heapreplace(heap, item)


def select_fast_train_indices():
    """
    One full TRAIN metadata scan.

    Returns:
      target_indices_by_type
      global_bg_indices       (<= BACKGROUND_STATS_MAX_CELLS)
      global_rest_indices     (subset of bg reservoir, <= DE_REST_MAX_CELLS)
      full_type_counts
    """
    candidate_set = set(marker_candidate_cell_types)
    type_heaps = defaultdict(list)
    global_heap = []
    counts = Counter()

    for i in tqdm(range(len(train_ds)), desc='select deterministic TRAIN reservoirs'):
        rec = train_ds[i]
        ct = normalize_spaces(rec.get(CELL_TYPE_FIELD, ''))
        cid = str(rec.get(CELL_ID_FIELD, i))
        counts[ct] += 1

        h_global = _hash32(f'global|{SEED}|{cid}|{i}')
        push_smallest_hash(global_heap, BACKGROUND_STATS_MAX_CELLS, h_global, i)

        if ct in candidate_set:
            h_type = _hash32(f'type|{SEED}|{ct}|{cid}|{i}')
            push_smallest_hash(type_heaps[ct], DE_TARGET_MAX_CELLS_PER_TYPE, h_type, i)

    target = {
        ct: sorted(idx for _, idx in heap)
        for ct, heap in type_heaps.items()
    }
    bg = sorted(idx for _, idx in global_heap)


    rest_ranked = sorted(
        bg,
        key=lambda i: _hash32(f'rest|{SEED}|{i}')
    )
    rest = rest_ranked[:int(DE_REST_MAX_CELLS)]

    return target, bg, rest, counts


def make_sparse_matrix_for_indices(indices):
    """Load only selected TRAIN cells and return log1p(normalize_total=1e4) CSR."""
    indices = sorted(set(int(x) for x in indices))
    row_map = {idx: r for r, idx in enumerate(indices)}

    rows = []
    cols = []
    vals = []
    obs_ct = []
    obs_id = []

    for idx in tqdm(indices, desc='load sampled TRAIN expression'):
        rec = train_ds[idx]
        mapped_genes, mapped_exprs = collator._extract_raw_mapped_gene_expr(dict(rec))

        gcols = []
        gvals = []
        for gid, val in zip(mapped_genes, mapped_exprs):
            if float(val) <= 0:
                continue
            col = gene_id_to_col.get(int(gid))
            if col is None:
                continue
            gcols.append(col)
            gvals.append(float(val))

        gcols, gvals = aggregate_duplicate_gene_counts(gcols, gvals)
        total = float(gvals.sum()) if len(gvals) else 0.0
        lognorm = np.log1p(gvals / total * 1e4).astype(np.float32) if total > 0 else np.asarray([], dtype=np.float32)

        r = row_map[idx]
        if len(gcols):
            rows.extend([r] * len(gcols))
            cols.extend(gcols.tolist())
            vals.extend(lognorm.tolist())

        obs_ct.append(normalize_spaces(rec.get(CELL_TYPE_FIELD, '')))
        obs_id.append(str(rec.get(CELL_ID_FIELD, idx)))

    X = sparse.csr_matrix(
        (
            np.asarray(vals, dtype=np.float32),
            (
                np.asarray(rows, dtype=np.int64),
                np.asarray(cols, dtype=np.int64),
            ),
        ),
        shape=(len(indices), len(all_gene_ids)),
        dtype=np.float32,
    )

    meta = pd.DataFrame({
        'dataset_index': indices,
        'cell_id': obs_id,
        'cell_type': obs_ct,
        'matrix_row': np.arange(len(indices), dtype=int),
    })
    return X, meta


def sparse_detection_fraction(X):
    return np.asarray((X > 0).mean(axis=0)).ravel().astype(np.float64)


def compute_global_gene_stats_from_rows(X, rows):
    rows = np.asarray(rows, dtype=np.int64)
    Xg = X[rows]
    mean_log = np.asarray(Xg.mean(axis=0)).ravel().astype(np.float64)
    det = sparse_detection_fraction(Xg)

    gs = pd.DataFrame({
        'gene_token_id': col_to_gene_id,
        'gene': gene_names,
        'mean_log_normalized_expression': mean_log,
        'detection_rate': det,
    })

    gs['expression_bin'] = -1
    gs['detection_bin'] = -1
    valid = gs['detection_rate'] > 0

    def safe_qbin(series, q):
        if len(series) == 0:
            return pd.Series(dtype=int)
        ranked = series.rank(method='average')
        try:
            return pd.qcut(
                ranked,
                q=min(int(q), max(1, int(series.nunique()))),
                labels=False,
                duplicates='drop',
            ).fillna(0).astype(int)
        except Exception:
            return pd.Series(np.zeros(len(series), dtype=int), index=series.index)

    gs.loc[valid, 'expression_bin'] = safe_qbin(
        gs.loc[valid, 'mean_log_normalized_expression'], MATCH_N_BINS
    )
    gs.loc[valid, 'detection_bin'] = safe_qbin(
        gs.loc[valid, 'detection_rate'], MATCH_N_BINS
    )
    gs['expression_bin'] = gs['expression_bin'].astype(int)
    gs['detection_bin'] = gs['detection_bin'].astype(int)
    return gs


def fast_wilcoxon_for_type(X, in_rows, out_rows, cell_type):
    in_rows = np.asarray(in_rows, dtype=np.int64)
    out_rows = np.asarray(out_rows, dtype=np.int64)

    Xin = X[in_rows]
    Xout = X[out_rows]
    n_in = Xin.shape[0]
    n_out = Xout.shape[0]

    pct_in_all = sparse_detection_fraction(Xin)
    tested_cols = np.where(pct_in_all >= float(MARKER_PCT_IN_MIN))[0]

    if len(tested_cols) == 0 or n_in < 2 or n_out < 2:
        return pd.DataFrame()


    pct_out = sparse_detection_fraction(Xout[:, tested_cols])

    u_parts = []
    p_parts = []
    mean_in_parts = []
    mean_out_parts = []

    for start in range(0, len(tested_cols), int(DE_GENE_BATCH_SIZE)):
        cols_batch = tested_cols[start:start + int(DE_GENE_BATCH_SIZE)]

        a = Xin[:, cols_batch].toarray().astype(np.float64, copy=False)
        b = Xout[:, cols_batch].toarray().astype(np.float64, copy=False)


        res = mannwhitneyu(
            a,
            b,
            axis=0,
            alternative='greater',
            method='asymptotic',
            use_continuity=True,
        )

        u_parts.append(np.asarray(res.statistic, dtype=np.float64))
        p_parts.append(np.asarray(res.pvalue, dtype=np.float64))


        mean_in_parts.append(np.expm1(a).mean(axis=0))
        mean_out_parts.append(np.expm1(b).mean(axis=0))

    U = np.concatenate(u_parts)
    p = np.concatenate(p_parts)
    mean_in = np.concatenate(mean_in_parts)
    mean_out = np.concatenate(mean_out_parts)

    p_adj = bh_adjust_full_universe(
        p_tested=p,
        tested_cols=tested_cols,
        n_total_genes=X.shape[1],
    )

    eps = 1e-8
    logfc = np.log2((mean_in + eps) / (mean_out + eps))
    auc_effect = U / float(n_in * n_out)

    d = pd.DataFrame({
        'cell_type': cell_type,
        'gene_token_id': col_to_gene_id[tested_cols].astype(int),
        'gene': gene_names[tested_cols],
        'mannwhitney_u': U,
        'auc_effect': auc_effect,
        'p_value': p,
        'p_adj': p_adj,
        'log2fc': logfc,
        'pct_in': pct_in_all[tested_cols],
        'pct_out': pct_out,
        'n_in': int(n_in),
        'n_out': int(n_out),
    })
    return d





if marker_all_path.exists() and marker_path.exists() and train_gene_stats_path.exists() and not FORCE_RERUN_MARKERS:
    marker_all_df = pd.read_csv(marker_all_path, compression='gzip')
    marker_df = pd.read_csv(marker_path)
    gene_stats_df = pd.read_csv(train_gene_stats_path)
    print('Loaded cached FAST Wilcoxon marker reference.')
else:
    target_indices_by_type, bg_indices, rest_indices, full_train_counts = select_fast_train_indices()

    marker_testable_cell_types = sorted([
        ct for ct in marker_candidate_cell_types
        if int(full_train_counts.get(ct, 0)) >= int(MIN_TRAIN_CELLS_PER_TYPE)
        and len(target_indices_by_type.get(ct, [])) >= 2
    ])


    selected_indices = set(bg_indices)
    for ct in marker_testable_cell_types:
        selected_indices.update(target_indices_by_type.get(ct, []))

    Xs, sampled_meta = make_sparse_matrix_for_indices(selected_indices)
    index_to_row = dict(zip(sampled_meta['dataset_index'].astype(int), sampled_meta['matrix_row'].astype(int)))

    bg_rows = [index_to_row[i] for i in bg_indices if i in index_to_row]
    rest_rows_global = [index_to_row[i] for i in rest_indices if i in index_to_row]

    gene_stats_df = compute_global_gene_stats_from_rows(Xs, bg_rows)
    gene_stats_df.to_csv(train_gene_stats_path, index=False)

    summary_rows = []
    frames = []

    for ct in tqdm(marker_testable_cell_types, desc='FAST one-vs-rest Wilcoxon by cell type'):
        in_rows = [index_to_row[i] for i in target_indices_by_type.get(ct, []) if i in index_to_row]
        out_rows = [r for r in rest_rows_global if sampled_meta.iloc[r]['cell_type'] != ct]

        d = fast_wilcoxon_for_type(Xs, in_rows, out_rows, ct)
        if len(d):
            frames.append(d)

        summary_rows.append({
            'cell_type': ct,
            'n_train_full': int(full_train_counts.get(ct, 0)),
            'n_target_de': int(len(in_rows)),
            'n_rest_de': int(len(out_rows)),
            'n_tested_genes': int(len(d)),
        })

    pd.DataFrame(summary_rows).to_csv(selection_path, index=False)

    marker_all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    marker_all_df.to_csv(marker_all_path, index=False, compression='gzip')

    keep = (
        marker_all_df['p_adj'].notna()
        & (marker_all_df['p_adj'] < float(MARKER_FDR_MAX))
        & marker_all_df['log2fc'].notna()
        & (marker_all_df['log2fc'] > float(MARKER_LOGFC_MIN))
        & (marker_all_df['pct_in'] >= float(MARKER_PCT_IN_MIN))
    )
    if REQUIRE_PCT_IN_GT_OUT:
        keep &= marker_all_df['pct_out'].notna() & (marker_all_df['pct_in'] > marker_all_df['pct_out'])

    passed = marker_all_df[keep].sort_values(
        ['cell_type', 'auc_effect', 'log2fc', 'p_adj'],
        ascending=[True, False, False, True],
    )

    marker_df = passed.groupby('cell_type', group_keys=False).head(int(MARKER_TOP_N)).copy()
    marker_df['rank'] = marker_df.groupby('cell_type').cumcount() + 1
    marker_df.to_csv(marker_path, index=False)

marker_counts = marker_df.groupby('cell_type').size().sort_values().rename('n_reference_markers')
valid_marker_types = set(marker_counts[marker_counts >= int(MIN_MATCHABLE_MARKERS_PER_CELL)].index)
marker_df = marker_df[marker_df['cell_type'].isin(valid_marker_types)].copy()
marker_sets = {ct: set(g['gene_token_id'].astype(int)) for ct, g in marker_df.groupby('cell_type')}

gene_stats_lookup = (
    gene_stats_df.set_index('gene_token_id')[[
        'expression_bin',
        'detection_bin',
        'mean_log_normalized_expression',
        'detection_rate',
    ]].to_dict('index')
)

print('VAL exact-span candidates:', len(marker_candidate_cell_types))
print('cell types with FAST Wilcoxon markers:', len(marker_counts))
print('cell types with >=5 reference markers:', len(marker_sets))
display(marker_counts.head(40))
display(marker_df.head(40))



## 9. Qualitative annotation generation sanity check

这一部分只回答：

> 模型加载、encoder、Annotation Decoder 是否能正常自由生成 annotation？

它**不是主结果，也不用于 cross-attention 统计**。

主机制分析仍然 teacher-force held-out test 的原始 annotation，因为：

- cell-type span 能被精确定位；
- 不需要解析生成文本；
- 不会因为生成 wording 改写导致 token span 不可比较。


In [ ]:





def move_tensor_batch_to_device(batch):
    out = {}
    for k, v in batch.items():
        out[k] = (
            v.to(DEVICE, non_blocking=True)
            if torch.is_tensor(v)
            else v
        )
    return out

def global_text_ids_to_text(ids):
    bert_tokens = []

    prefix = str(tokenizer.text_token_prefix)
    suffix = str(tokenizer.token_suffix)

    for gid in ids:
        gid = int(gid)

        if gid == int(tokenizer.eos_token_id):
            break

        tok = tokenizer.global_id_to_token.get(gid, "")

        if (
            tok.startswith(prefix)
            and tok.endswith(suffix)
        ):
            end = -len(suffix) if suffix else None
            bert_tokens.append(tok[len(prefix):end])

    if not bert_tokens:
        return ""

    return tokenizer.text_tokenizer.convert_tokens_to_string(
        bert_tokens
    )

def autocast_ctx():
    if DEVICE.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )
    return nullcontext()

@torch.inference_mode()
def greedy_generate_annotation(rec, max_new_tokens=64):
    cpu_batch = collator([dict(rec)])
    batch = move_tensor_batch_to_device(cpu_batch)

    enc, _ = model.encode(
        encoder_input_gene_ids=batch["encoder_input_gene_ids"],
        encoder_input_values=batch["encoder_input_values"],
        encoder_key_padding_mask=batch["encoder_key_padding_mask"],
        return_last_attn=False,
    )

    prefix_target = (
        [int(tokenizer.task_token_id)]
        + tokenizer.encode_task_name_to_global_ids("annotation")
        + [int(tokenizer.start_answer_token_id)]
    )

    decoder_ids = (
        [int(tokenizer.bos_token_id)]
        + prefix_target
    )

    generated = []

    for _ in range(int(max_new_tokens)):
        dec = torch.tensor(
            [decoder_ids],
            dtype=torch.long,
            device=DEVICE,
        )

        mask = torch.ones_like(dec, dtype=torch.long)

        with autocast_ctx():
            hidden, _ = model.decode_annotation(
                decoder_input_ids=dec,
                encoder_outputs=enc,
                decoder_attention_mask=mask,
                encoder_key_padding_mask=batch[
                    "encoder_key_padding_mask"
                ],
                return_last_cross_attn=False,
            )

            logits = model.annotation_lm_head(hidden)

        next_id = int(
            logits[0, -1]
            .float()
            .argmax()
            .item()
        )

        if next_id == int(tokenizer.eos_token_id):
            break

        generated.append(next_id)
        decoder_ids.append(next_id)

        if len(decoder_ids) >= int(model.max_decoder_length):
            break

    return global_text_ids_to_text(generated)

if RUN_GENERATION_SANITY:
    generation_rows = []


    used_types = set()

    for r in span_df[
        span_df["span_success"]
        & span_df["cell_type"].isin(marker_sets.keys())
    ].itertuples(index=False):

        if r.cell_type in used_types:
            continue

        rec = val_ds[int(r.dataset_index)]

        pred_text = greedy_generate_annotation(
            rec,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
        )

        true_ct = normalize_spaces(rec[CELL_TYPE_FIELD])

        generation_rows.append({
            "dataset_index": int(r.dataset_index),
            "cell_id": str(rec[CELL_ID_FIELD]),
            "cell_type": true_ct,
            "target_annotation": normalize_spaces(
                rec[ANNOTATION_FIELD]
            ),
            "generated_annotation": pred_text,
            "generated_contains_celltype_exact": bool(
                re.search(
                    re.escape(true_ct),
                    pred_text,
                    flags=re.IGNORECASE,
                )
            ),
        })

        used_types.add(r.cell_type)

        if len(generation_rows) >= int(N_GENERATION_EXAMPLES):
            break

    generation_df = pd.DataFrame(generation_rows)

    generation_df.to_csv(
        OUT_DIR / "annotation_generation_examples.csv",
        index=False,
    )

    display(
        generation_df[
            [
                "cell_type",
                "target_annotation",
                "generated_annotation",
                "generated_contains_celltype_exact",
            ]
        ]
    )



# 10. 主机制分析：cell-type span cross-attention

对每一个 eligible held-out VAL cell：

```text
完整原始 annotation
        ↓ teacher forcing
cell-type span query positions
        ↓
最后一层 Annotation Decoder cross-attention
        ↓
encoder genes
```

Attention 聚合：

```text
mean over:
    decoder attention heads
    ×
    all token positions in the cell-type span
```

然后对当前 cell 的有效 encoder genes 做 **within-cell percentile rank**。

默认：

- 排除 `<cls>` / `<pad>`
- 只保留真正 gene tokens
- 如果保持原始 10% MLM masking，则主统计排除本次被 expression-mask 的 positions
- marker 必须是：
  - 当前 cell type 的 train-derived marker
  - 且实际进入当前 encoder sequence


In [ ]:





def stable_seed(text, extra=0):
    return (
        int(SEED)
        + int(zlib.crc32(str(text).encode("utf-8")) & 0xFFFFFFFF)
        + int(extra)
    ) % (2**32 - 1)

def attention_percentile(values):
    values = np.asarray(values, dtype=np.float64)

    if len(values) <= 1:
        return np.ones_like(values, dtype=np.float64)

    r = rankdata(values, method="average")

    return (r - 1.0) / (len(values) - 1.0)

def compute_cell_matched_score(gene_df, cell_type):
    """
    对当前 cell：
      1. 选 train-derived markers
      2. 对每个 marker，找 expression_bin + detection_bin 相同的 non-marker
      3. marker score 只使用能够成功匹配 background 的 marker
      4. background 重复随机匹配 MATCH_REPEATS 次
    """
    marker_set = marker_sets.get(cell_type, set())

    if not marker_set:
        return None

    g = gene_df.copy()
    g["gene_token_id"] = g["gene_token_id"].astype(int)

    marker_rows = g[
        g["gene_token_id"].isin(marker_set)
    ].copy()

    nonmarker_rows = g[
        ~g["gene_token_id"].isin(marker_set)
    ].copy()

    if len(marker_rows) == 0 or len(nonmarker_rows) == 0:
        return None

    pools = defaultdict(list)

    for r in nonmarker_rows.itertuples(index=False):
        st = gene_stats_lookup.get(int(r.gene_token_id))

        if st is None:
            continue

        eb = int(st["expression_bin"])
        db = int(st["detection_bin"])

        if eb < 0 or db < 0:
            continue

        pools[(eb, db)].append(
            float(r.attention_percentile)
        )

    matchable_marker_attention = []
    match_keys = []

    for r in marker_rows.itertuples(index=False):
        st = gene_stats_lookup.get(int(r.gene_token_id))

        if st is None:
            continue

        eb = int(st["expression_bin"])
        db = int(st["detection_bin"])

        key = (eb, db)

        if eb < 0 or db < 0:
            continue

        if key not in pools or len(pools[key]) == 0:
            continue

        matchable_marker_attention.append(
            float(r.attention_percentile)
        )
        match_keys.append(key)

    if (
        len(matchable_marker_attention)
        < int(MIN_MATCHABLE_MARKERS_PER_CELL)
    ):
        return None

    marker_score = float(
        np.median(matchable_marker_attention)
    )

    rng = np.random.default_rng(
        stable_seed(str(g["cell_id"].iloc[0]))
    )

    bg_repeat_scores = []

    for _ in range(int(MATCH_REPEATS)):
        vals = []

        for key in match_keys:
            pool = pools[key]
            vals.append(
                float(
                    pool[int(rng.integers(0, len(pool)))]
                )
            )

        bg_repeat_scores.append(
            float(np.median(vals))
        )

    background_score = float(
        np.mean(bg_repeat_scores)
    )

    return {
        "cell_id": str(g["cell_id"].iloc[0]),
        "cell_type": cell_type,
        "n_valid_encoder_genes": int(len(g)),
        "n_markers_present": int(len(marker_rows)),
        "n_markers_matchable": int(
            len(matchable_marker_attention)
        ),
        "marker_attention": marker_score,
        "background_attention": background_score,
        "delta_attention": float(
            marker_score - background_score
        ),
        "n_background_repeats": int(
            len(bg_repeat_scores)
        ),
    }

def save_attention_part(df, part_no, parts_dir):
    parquet_path = parts_dir / f"part_{part_no:06d}.parquet"

    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path

    except Exception:
        csv_path = parts_dir / f"part_{part_no:06d}.csv.gz"
        df.to_csv(
            csv_path,
            index=False,
            compression="gzip",
        )
        return csv_path


In [ ]:





parts_dir = OUT_DIR / "celltype_span_attention_parts"
parts_dir.mkdir(parents=True, exist_ok=True)

cell_level_path = OUT_DIR / "celltype_marker_attention_cell_level.csv"

existing_parts = sorted(
    list(parts_dir.glob("part_*.parquet"))
    + list(parts_dir.glob("part_*.csv.gz"))
)

if (
    cell_level_path.exists()
    and existing_parts
    and (not FORCE_RERUN_ATTENTION)
):
    cell_level_df = pd.read_csv(cell_level_path)
    print("Loaded cached attention results.")

else:

    for p in existing_parts:
        p.unlink()

    eligible = span_df[
        span_df["span_success"]
        & span_df["cell_type"].isin(marker_sets.keys())
        ].copy()

    print("eligible VAL cells:", len(eligible))
    print("eligible cell types:", eligible["cell_type"].nunique())

    compact_rows = []
    part_no = 0

    records_meta = eligible[
        [
            "dataset_index",
            "cell_id",
            "cell_type",
        ]
    ].to_records(index=False)

    for start in tqdm(
        range(0, len(records_meta), ATTENTION_BATCH_SIZE),
        desc="cell-type span cross-attention",
    ):
        chunk = records_meta[
            start:start + ATTENTION_BATCH_SIZE
        ]

        records = [
            val_ds[int(x.dataset_index)]
            for x in chunk
        ]


        span_infos = [
            locate_celltype_span(
                rec[CELL_TYPE_FIELD],
                rec[ANNOTATION_FIELD],
            )
            for rec in records
        ]


        cpu_batch = collator(
            [dict(x) for x in records]
        )

        if len(cpu_batch["cell_keys"]) != len(records):
            raise RuntimeError(
                "Annotation-only collator 输出行数与输入 records 不一致。"
            )

        batch = move_tensor_batch_to_device(cpu_batch)

        with torch.inference_mode():
            with autocast_ctx():
                outputs = model(
                    encoder_input_gene_ids=batch[
                        "encoder_input_gene_ids"
                    ],
                    encoder_input_values=batch[
                        "encoder_input_values"
                    ],
                    encoder_key_padding_mask=batch[
                        "encoder_key_padding_mask"
                    ],
                    decoder_input_ids=batch[
                        "decoder_input_ids"
                    ],
                    decoder_attention_mask=batch[
                        "decoder_attention_mask"
                    ],
                    decoder_task_ids=batch[
                        "decoder_task_ids"
                    ],
                    return_encoder_gene_logits=False,
                    return_last_cross_attn=True,
                )

        cross = outputs[
            "annotation_last_cross_attn"
        ].detach().float().cpu()


        if cross.ndim != 4:
            raise RuntimeError(
                f"annotation cross-attention shape 异常: {tuple(cross.shape)}"
            )

        target_gene_ids = cpu_batch[
            "encoder_target_gene_ids"
        ].cpu()

        encoder_kpm = cpu_batch[
            "encoder_key_padding_mask"
        ].cpu()

        expr_masks = cpu_batch[
            "expr_masks"
        ].cpu()

        batch_gene_rows = []

        for b, (rec, sinfo) in enumerate(
            zip(records, span_infos)
        ):
            if not sinfo.get("success", False):
                continue

            qpos = torch.tensor(
                sinfo["decoder_query_positions"],
                dtype=torch.long,
            )

            if int(qpos.max()) >= cross.shape[2]:
                continue


            gene_att = (
                cross[b, :, qpos, :]
                .mean(dim=(0, 1))
                .numpy()
            )

            gids = target_gene_ids[b].numpy()
            pads = encoder_kpm[b].numpy().astype(bool)
            masked_expr = expr_masks[b].numpy().astype(bool)

            valid_gid = []
            valid_att = []
            valid_masked_flag = []

            for pos, (gid, att, is_pad, is_expr_masked) in enumerate(
                zip(gids, gene_att, pads, masked_expr)
            ):
                gid = int(gid)

                if is_pad:
                    continue

                if gid == int(tokenizer.cls_token_id):
                    continue

                if not tokenizer.global_id_is_gene(gid):
                    continue

                if (
                    EXCLUDE_EXPR_MASKED_GENES_FROM_ATTENTION
                    and bool(is_expr_masked)
                ):
                    continue

                valid_gid.append(gid)
                valid_att.append(float(att))
                valid_masked_flag.append(
                    bool(is_expr_masked)
                )

            if len(valid_gid) == 0:
                continue

            pct = attention_percentile(valid_att)

            cell_id = str(rec[CELL_ID_FIELD])
            cell_type = normalize_spaces(
                rec[CELL_TYPE_FIELD]
            )
            ann = normalize_spaces(
                rec[ANNOTATION_FIELD]
            )
            marker_set = marker_sets.get(
                cell_type,
                set(),
            )

            rows_this_cell = []

            for gid, att, p, was_masked in zip(
                valid_gid,
                valid_att,
                pct,
                valid_masked_flag,
            ):
                rows_this_cell.append({
                    "cell_id": cell_id,
                    "cell_type": cell_type,
                    "natural_language_annotation": ann,
                    "matched_celltype_text": sinfo[
                        "matched_surface_text"
                    ],
                    "n_celltype_tokens": sinfo[
                        "n_celltype_tokens"
                    ],
                    "gene_token_id": int(gid),
                    "gene": gene_name(gid),
                    "attention_raw": float(att),
                    "attention_percentile": float(p),
                    "is_train_reference_marker": bool(
                        gid in marker_set
                    ),
                    "expr_was_mlm_masked": bool(
                        was_masked
                    ),
                })

            cell_gene_df = pd.DataFrame(
                rows_this_cell
            )

            compact = compute_cell_matched_score(
                cell_gene_df,
                cell_type,
            )

            if compact is not None:
                compact_rows.append(compact)

            if SAVE_GENE_LEVEL_ATTENTION:
                batch_gene_rows.extend(
                    rows_this_cell
                )

        if (
            SAVE_GENE_LEVEL_ATTENTION
            and batch_gene_rows
        ):
            save_attention_part(
                pd.DataFrame(batch_gene_rows),
                part_no,
                parts_dir,
            )
            part_no += 1

    cell_level_df = pd.DataFrame(compact_rows)
    cell_level_df.to_csv(
        cell_level_path,
        index=False,
    )

print("evaluable cells:", len(cell_level_df))
print(
    "evaluable cell types:",
    cell_level_df["cell_type"].nunique()
    if len(cell_level_df) else 0
)

display(cell_level_df.head())


In [ ]:






if len(cell_level_df)==0:
    raise RuntimeError('No evaluable VAL cells.')

celltype_level_all_df=(
    cell_level_df.groupby('cell_type',as_index=False).agg(
        n_cells=('cell_id','nunique'),
        median_valid_encoder_genes=('n_valid_encoder_genes','median'),
        median_markers_present=('n_markers_present','median'),
        median_markers_matchable=('n_markers_matchable','median'),
        marker_attention=('marker_attention','median'),
        background_attention=('background_attention','median'),
        delta_attention=('delta_attention','median'),
    )
)
celltype_level_all_df.to_csv(OUT_DIR/'celltype_marker_attention_celltype_level_all.csv',index=False)

celltype_level_df=(celltype_level_all_df[celltype_level_all_df['n_cells']>=int(MIN_VAL_EVALUABLE_CELLS_PER_TYPE)]
                   .copy().sort_values('delta_attention',ascending=False).reset_index(drop=True))
celltype_level_df.to_csv(OUT_DIR/'celltype_marker_attention_celltype_level_final.csv',index=False)

print('Cell types before final filter:',len(celltype_level_all_df))
print('Cell types entering paired statistics:',len(celltype_level_df))
print('Minimum evaluable VAL cells/type:',MIN_VAL_EVALUABLE_CELLS_PER_TYPE)

if len(celltype_level_df)<2:
    raise RuntimeError('Final evaluable cell types < 2.')

w=wilcoxon(celltype_level_df['marker_attention'],celltype_level_df['background_attention'],alternative=WILCOXON_ALTERNATIVE,zero_method='wilcox')
stats_df=pd.DataFrame({
    'n_cell_types':[len(celltype_level_df)],
    'n_positive_cell_types':[(celltype_level_df['delta_attention']>0).sum()],
    'fraction_positive_cell_types':[(celltype_level_df['delta_attention']>0).mean()],
    'median_marker_attention':[np.median(celltype_level_df['marker_attention'])],
    'median_background_attention':[np.median(celltype_level_df['background_attention'])],
    'median_delta_attention':[np.median(celltype_level_df['delta_attention'])],
    'wilcoxon_statistic':[float(w.statistic)],
    'p_value':[float(w.pvalue)],
    'alternative':[WILCOXON_ALTERNATIVE],
})
stats_df.to_csv(OUT_DIR/'celltype_marker_attention_statistics.csv',index=False)
display(celltype_level_df)
display(stats_df)


In [ ]:







import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(4.4, 5.0)
)

for r in celltype_level_df.itertuples(
    index=False
):
    ax.plot(
        [0, 1],
        [
            r.background_attention,
            r.marker_attention,
        ],
        marker="o",
        linewidth=1.0,
        alpha=0.75,
    )

ax.set_xlim(-0.25, 1.25)
ax.set_xticks([0, 1])
ax.set_xticklabels([
    "Matched\nbackground",
    "Cell-type\nmarkers",
])

ax.set_ylabel(
    "Cell-type-span cross-attention percentile"
)

n_pos = int(
    (
        celltype_level_df[
            "delta_attention"
        ] > 0
    ).sum()
)
n_types = int(len(celltype_level_df))
p = float(stats_df.loc[0, "p_value"])

ax.text(
    0.02,
    0.98,
    f"{n_pos}/{n_types} cell types ↑\n"
    f"Paired Wilcoxon P = {p:.3g}",
    transform=ax.transAxes,
    ha="left",
    va="top",
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

fig.savefig(
    OUT_DIR / "fig5b_celltype_span_marker_attention.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUT_DIR / "fig5b_celltype_span_marker_attention.svg",
    bbox_inches="tight",
)

fig.savefig(
    OUT_DIR / "fig5b_celltype_span_marker_attention.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:








def list_attention_parts():
    return sorted(
        list(parts_dir.glob("part_*.parquet"))
        + list(parts_dir.glob("part_*.csv.gz"))
    )

def read_attention_part(path):
    path = Path(path)

    if path.suffix == ".parquet":
        return pd.read_parquet(path)

    return pd.read_csv(
        path,
        compression="gzip",
    )

if SAVE_GENE_LEVEL_ATTENTION and list_attention_parts():
    representative_type = (
        celltype_level_df
        .sort_values(
            "n_cells",
            ascending=False,
        )
        .iloc[0]["cell_type"]
    )

    pieces = []

    for pth in tqdm(
        list_attention_parts(),
        desc="representative markers",
    ):
        part = read_attention_part(pth)

        sub = part[
            (part["cell_type"] == representative_type)
            & part[
                "is_train_reference_marker"
            ].astype(bool)
        ]

        if len(sub):
            pieces.append(
                sub[
                    [
                        "cell_id",
                        "gene_token_id",
                        "gene",
                        "attention_percentile",
                    ]
                ]
            )

    if pieces:
        rep = pd.concat(
            pieces,
            ignore_index=True,
        )

        rep_summary = (
            rep.groupby(
                ["gene_token_id", "gene"],
                as_index=False,
            )
            .agg(
                median_attention_percentile=(
                    "attention_percentile",
                    "median",
                ),
                n_cells=(
                    "cell_id",
                    "nunique",
                ),
            )
            .sort_values(
                [
                    "median_attention_percentile",
                    "n_cells",
                ],
                ascending=[False, False],
            )
            .head(15)
        )

        display(rep_summary)

        fig, ax = plt.subplots(
            figsize=(4.6, 4.4)
        )

        y = np.arange(len(rep_summary))

        ax.barh(
            y,
            rep_summary[
                "median_attention_percentile"
            ],
        )

        ax.set_yticks(y)
        ax.set_yticklabels(
            rep_summary["gene"]
        )
        ax.invert_yaxis()

        ax.set_xlabel(
            "Median cell-type-span\ncross-attention percentile"
        )
        ax.set_title(
            representative_type
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        fig.tight_layout()

        fig.savefig(
            OUT_DIR
            / "fig5b_representative_marker_genes.pdf",
            bbox_inches="tight",
        )

        fig.savefig(
            OUT_DIR
            / "fig5b_representative_marker_genes.svg",
            bbox_inches="tight",
        )

        plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import wilcoxon





mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["axes.unicode_minus"] = False


mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"





TITLE_SIZE = 9
LABEL_SIZE = 8
TICK_SIZE = 8
ANNOTATION_SIZE = 8





CSV_PATH = OUT_DIR / 'celltype_marker_attention_celltype_level_final.csv'

df = pd.read_csv(CSV_PATH)

marker = df["marker_attention"].to_numpy()
background = df["background_attention"].to_numpy()





stat, p = wilcoxon(
    marker,
    background,
    alternative="two-sided"
)

n_types = len(df)
n_positive = int(
    (df["marker_attention"] > df["background_attention"]).sum()
)

marker_median = float(np.median(marker))
background_median = float(np.median(background))
delta_median = float(np.median(marker - background))





fig, ax = plt.subplots(figsize=(4.0, 4.8))

x_marker = 0
x_background = 1

marker_color = "#8BC34A"
background_color = "#BDBDBD"





for _, row in df.iterrows():
    ax.plot(
        [x_marker, x_background],
        [row["marker_attention"], row["background_attention"]],
        color="0.70",
        linewidth=0.5,
        alpha=0.22,
        zorder=1,
    )





vp1 = ax.violinplot(
    marker,
    positions=[x_marker],
    widths=0.52,
    showmeans=False,
    showmedians=False,
    showextrema=False,
)

vp2 = ax.violinplot(
    background,
    positions=[x_background],
    widths=0.52,
    showmeans=False,
    showmedians=False,
    showextrema=False,
)

for body in vp1["bodies"]:
    body.set_facecolor(marker_color)
    body.set_edgecolor("black")
    body.set_linewidth(0.9)
    body.set_alpha(0.9)

for body in vp2["bodies"]:
    body.set_facecolor(background_color)
    body.set_edgecolor("black")
    body.set_linewidth(0.9)
    body.set_alpha(0.9)





bp = ax.boxplot(
    [marker, background],
    positions=[x_marker, x_background],
    widths=0.18,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(
        color="black",
        linewidth=1.2,
    ),
    whiskerprops=dict(
        color="black",
        linewidth=0.8,
    ),
    capprops=dict(
        color="black",
        linewidth=0.8,
    ),
    boxprops=dict(
        edgecolor="black",
        linewidth=0.8,
    ),
)

bp["boxes"][0].set_facecolor(marker_color)
bp["boxes"][0].set_alpha(0.75)

bp["boxes"][1].set_facecolor(background_color)
bp["boxes"][1].set_alpha(0.75)





ax.scatter(
    [x_marker],
    [marker_median],
    s=36,
    facecolor="white",
    edgecolor="black",
    linewidth=0.9,
    zorder=5,
)

ax.scatter(
    [x_background],
    [background_median],
    s=36,
    facecolor="white",
    edgecolor="black",
    linewidth=0.9,
    zorder=5,
)





ymax = max(marker.max(), background.max())
y1 = ymax + 0.025
y2 = ymax + 0.045

ax.plot(
    [x_marker, x_marker, x_background, x_background],
    [y1, y2, y2, y1],
    color="black",
    linewidth=1.0,
)

if p < 1e-4:
    p_text = "****"
elif p < 1e-3:
    p_text = "***"
elif p < 1e-2:
    p_text = "**"
elif p < 0.05:
    p_text = "*"
else:
    p_text = "ns"

ax.text(
    0.5,
    y2 + 0.008,
    p_text,
    ha="center",
    va="bottom",
    fontsize=TITLE_SIZE,
    fontweight="bold",
    fontfamily="Arial",
)





ax.set_xticks([0, 1])

ax.set_xticklabels(
    [
        "Marker\ngenes",
        "Matched\nbackground\ngenes",
    ],
    fontsize=TICK_SIZE,
    fontfamily="Arial",
)


ax.get_xticklabels()[0].set_color(marker_color)

ax.set_ylabel(
    "Cell-type-span\ncross-attention percentile",
    fontsize=LABEL_SIZE,
    fontfamily="Arial",
)

ax.set_title(
    f"Paired across cell types\n"
    f"(n = {n_types} fine cell types)",
    fontsize=TITLE_SIZE,
    fontweight="bold",
    fontfamily="Arial",
    pad=8,
)





ax.text(
    0.98,
    0.02,
    f"{n_positive}/{n_types} ↑\n"
    f"P = {p:.1e}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=ANNOTATION_SIZE,
    fontfamily="Arial",
)





ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)

ax.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
    width=0.8,
    length=3.5,
)

for label in ax.get_yticklabels():
    label.set_fontfamily("Arial")

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, y2 + 0.09)

plt.tight_layout()





plt.savefig(
    "Fig5b_marker_vs_background_attention_Arial.pdf",
    bbox_inches="tight",
)

plt.savefig(
    "Fig5b_marker_vs_background_attention_Arial.svg",
    bbox_inches="tight",
)

plt.savefig(
    "Fig5b_marker_vs_background_attention_Arial.png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()


# 15. 主要输出 — Fast version

### 核心 marker 文件

- `train_fast_wilcoxon_sampling_summary.csv`
- `train_fast_wilcoxon_markers_all.csv.gz`
- `train_fast_wilcoxon_reference_markers_topN.csv`
- `train_sampled_global_gene_stats.csv`

### Attention

- `celltype_marker_attention_cell_level.csv`
- `celltype_marker_attention_celltype_level_all.csv`
- `celltype_marker_attention_celltype_level_final.csv`
- `celltype_marker_attention_statistics.csv`

### Figure

- `fig5b_celltype_span_marker_attention.pdf`
- `fig5b_celltype_span_marker_attention.svg`
- `fig5b_celltype_span_marker_attention.png`

## Fast Wilcoxon Methods 定义

Reference markers are derived exclusively from the training partition using a deterministic subsampled one-versus-rest Wilcoxon rank-sum analysis. For each target cell type, up to 200 training cells are compared with a fixed global reference reservoir of up to 1,000 training cells, excluding cells of the target type. Genes detected in at least 10% of target cells are tested using the Mann–Whitney U statistic, which is equivalent to the Wilcoxon rank-sum test for two independent samples. Benjamini–Hochberg correction is applied over the complete gene universe, with untested genes assigned P=1. Genes are retained at adjusted P<0.05, log2 fold-change>0.25, target detection frequency≥10%, and target detection frequency greater than background. The top 30 genes are ranked by U-derived AUC effect size and log2 fold-change.

### 为什么默认不保存 gene-level attention

主分析只需要每个 cell 的 marker/background summary。关闭 `SAVE_GENE_LEVEL_ATTENTION` 可以显著减少磁盘 I/O；如果后续要画代表性 marker inset，再单独对少数 cell types 重跑即可。


In [ ]:





files = sorted(
    x for x in OUT_DIR.rglob("*")
    if x.is_file()
)

inventory = pd.DataFrame({
    "file": [
        str(x.relative_to(OUT_DIR))
        for x in files
    ],
    "size_MB": [
        round(
            x.stat().st_size / 1024**2,
            3,
        )
        for x in files
    ],
})

display(inventory)

print("\n=== Span coverage ===")
display(span_summary)

print("\n=== Main mechanism statistics ===")
display(stats_df)
